# Importing the Packages needed For Data Cleaning and Data Visualization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as py
import plotly.io as pio

In [ ]:
pio.renderers.default = 'notebook_connected'

In [ ]:
import os

In [ ]:
os.listdir(r'C:\Users\2097770\OneDrive - Cognizant\Desktop\Data Analytics Course\Data Analystics Projects\Uber\Datasets')

In [ ]:
uber_data = pd.read_csv(r'C:\Users\2097770\OneDrive - Cognizant\Desktop\Data Analytics Course\Data Analystics Projects\Uber\Datasets\uber-raw-data-janjune-15_sample.csv')

In [ ]:
uber_data

In [ ]:
type(uber_data)

In [ ]:
uber_data.shape

In [ ]:
uber_data.duplicated()

In [ ]:
uber_data.duplicated()

In [ ]:
uber_data[uber_data.duplicated(keep = False)].sort_values(by = uber_data.columns.to_list())

In [ ]:
uber_data.drop_duplicates(inplace = True)

In [ ]:
uber_data.shape

In [ ]:
uber_data.isnull()

In [ ]:
uber_data.isnull().sum()

In [ ]:
uber_data.dtypes

In [ ]:
uber_data["Pickup_date"] = pd.to_datetime(uber_data["Pickup_date"])

In [ ]:
uber_data.dtypes

In [ ]:
uber_data["Pickup_date"][0]

# which month have max uber pickups in new york city

In [ ]:
uber_data["Month"] = uber_data["Pickup_date"].dt.month_name()
uber_data["WeekDay"] = uber_data["Pickup_date"].dt.day_name()
uber_data["Day"] = uber_data["Pickup_date"].dt.day
uber_data["Hour"] = uber_data["Pickup_date"].dt.hour
uber_data["Minute"] = uber_data["Pickup_date"].dt.minute

In [ ]:
uber_data["Month"]

In [ ]:
uber_data["Month"].value_counts(sort = True)

In [ ]:
month_order = ['January','February','March','April','May','June']

In [ ]:
uber_data["Month"].value_counts().reindex(month_order).plot(
    kind="bar", 
    color="skyblue", 
    title="Total Uber Pickups by Month (Jan-June)",
    xlabel="Month",
    ylabel="Number of Pickups"
)

In [ ]:
uber_data.head()

# Uber Pickup by Month and weekday

In [ ]:
pivot = pd.crosstab(index = uber_data["Month"] , columns = uber_data["WeekDay"])

In [ ]:
pivot

In [ ]:
month_order = ['January','February','March','April','May','June']

weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']



In [ ]:
pivot_sorted = pivot.reindex(index = month_order).reindex(columns = weekday_order)

In [ ]:
pivot_sorted

In [ ]:
fig = py.bar(pivot_sorted , 
      x = pivot_sorted.index,
      y = pivot_sorted.columns,
      barmode = "group",
      title = "Uber Pickups by Month & Weekday")

In [ ]:
fig.update_layout(
    xaxis_title = "Month",
    yaxis_title = "Number of Pickups",
    legend_title = "WeekDay" 
)

# Uber Pickup by Week Days

In [ ]:
weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

In [ ]:
uber_data["WeekDay"].value_counts()

In [ ]:
uber_data["WeekDay"].value_counts().reindex(weekday_order).plot(
    kind = "bar",
    title = 'Uber Pickup by WeekDays',
    color = 'skyblue',
    xlabel = 'Week Days',
    ylabel = 'Number of Pickups'
)

# Hourly Uber Rush by WeekDays

In [ ]:
uber_data.head(4)

In [ ]:
hourly_rush = pd.crosstab(index = uber_data["WeekDay"] , columns = uber_data["Hour"])

In [ ]:
hourly_rush

In [ ]:
weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']


In [ ]:
hourly_rush = hourly_rush.reindex(index = weekday_order)

In [ ]:
hourly_rush

In [ ]:
hourly_rush = uber_data.groupby(['WeekDay','Hour'],as_index = False).size()

In [ ]:
hourly_rush

In [ ]:
weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']


In [ ]:
hourly_rush["WeekDay"] = pd.Categorical(hourly_rush["WeekDay"],categories = weekday_order , ordered = True)

In [ ]:
hourly_rush = hourly_rush.sort_values(['WeekDay','Hour'])

In [ ]:
hourly_rush

In [ ]:
hourly_fig = py.line(hourly_rush,
        x = "Hour",
        y = "size",
        color = "WeekDay",
        markers = True,
        title = "Hourly Uber Rush by Days"
       )

In [ ]:
hourly_fig

In [ ]:
hourly_fig.update_layout(xaxis_title = 'Hour of the Day',
                        yaxis_title = 'Number of Pickups',
                        legend_title = 'WeekDay',
                        template = 'plotly_white')

# Pareto Analysis

In [ ]:
uber_data.head(1)

In [ ]:
base_counts = uber_data['Dispatching_base_num'].value_counts()

In [ ]:
base_counts

In [ ]:
base_counts.sum()

In [ ]:
base_counts/base_counts.sum()

In [ ]:
(base_counts/base_counts.sum()).cumsum()

In [ ]:
base_df = base_counts.reset_index()

In [ ]:
base_df

In [ ]:
base_df.columns = ['Base','Trips']

In [ ]:
base_df

In [ ]:
base_df['Trip_Percentage'] = base_df['Trips']/base_df['Trips'].sum()

In [ ]:
base_df

In [ ]:
base_df['Cumulative_Percentage'] = (base_df['Trips']/base_df['Trips'].sum()).cumsum()

In [ ]:
base_df

In [ ]:
import plotly.graph_objs as go

In [ ]:
base_df = base_df.sort_values("Trips", ascending=False).reset_index(drop=True)

In [ ]:
base_df["Cumulative_Percentage"] = (base_df["Trips"].cumsum() / base_df["Trips"].sum()) * 100

In [ ]:
base_df

In [ ]:
pareto = go.Figure(
    data = [
        go.Bar(
            x = base_df["Base"],
            y = base_df["Trips"],
            name = "Trips",
            marker_color = "steelblue",
            yaxis = 'y1'
        ),
        go.Scatter(
            x = base_df["Base"],
            y = base_df["Cumulative_Percentage"],
            yaxis = 'y2',
            mode = 'lines+markers',
            name = 'Cumulative %',
            line = dict(color='red', width=2),
            marker = dict(size=6)
        )
    ],
    layout = go.Layout(
        title = "Pareto Chart - Uber Trips by Base",
        xaxis = dict(title = "Base"),
        yaxis = dict(
            title = "Number of Trips",
            showgrid = True
        ),
        yaxis2 = dict(
            
            title = "Cumulative %",
            overlaying = 'y',
            side = 'right',
            range = [0, 100],
            ticksuffix = '%',
            showgrid = False
        ),
        legend = dict(x=0.7, y=1.1, orientation='h')
    )
)

In [ ]:
pareto.add_hline(y = 80,yref = 'y2', line_dash = 'dash')

# Airport Demand Analysis !!

In [ ]:
uber_data['locationID']

In [ ]:
airport_Ids = [1,132,138]

In [ ]:
airport_df = uber_data[uber_data['locationID'].isin(airport_Ids)].copy()

In [ ]:
airport_df

In [ ]:
airport_map = {
    1 : 'EWR',
    132: 'JFK',
    138: 'LGA'
}

In [ ]:
airport_df['locationID'].map(airport_map)

In [ ]:
airport_df['airport_name'] = airport_df['locationID'].map(airport_map)

In [ ]:
airport_df

In [ ]:
airport_df['airport_name'].value_counts()

In [ ]:
airport_df['Hour'] = airport_df['Pickup_date'].dt.hour

In [ ]:
airport_df

In [ ]:
airport_hourly_rush = airport_df.groupby(['airport_name','Hour']).size().reset_index(name = 'trips')

In [ ]:
airport_hourly_rush

In [ ]:
airport_hourly_rush

In [ ]:
area = py.area(
    airport_hourly_rush,
    x = "Hour",
    y = "trips",
    color = 'airport_name',
    title = 'Airport Demand Pressure',
    labels = {'Hour' : 'Hours of the Day','trips' : 'Number of trips'}
)

In [ ]:
area

# Collecting Entire Data and make it ready for data analysis

In [ ]:
import os

In [ ]:
files = os.listdir(r'C:\Users\2097770\OneDrive - Cognizant\Desktop\Data Analytics Course\Data Analystics Projects\Uber\Datasets')

In [ ]:
files

In [ ]:
files_csv = ['uber-raw-data-jul14.csv',
 'uber-raw-data-jun14.csv',
 'uber-raw-data-may14.csv',
 'uber-raw-data-sep14.csv',
 'uber-raw-data-apr14.csv',
 'uber-raw-data-aug14.csv',
]

In [ ]:
final = pd.DataFrame()
path = r'C:\Users\2097770\OneDrive - Cognizant\Desktop\Data Analytics Course\Data Analystics Projects\Uber\Datasets'

for file in files_csv:
    current_df = pd.read_csv(path + '/' + file)
    final = pd.concat([current_df,final])

In [ ]:
final.shape

In [ ]:
final[final.duplicated()].head()

In [ ]:
final.duplicated().sum()

In [ ]:
final.drop_duplicates(inplace = True)

In [ ]:
final.duplicated().sum()

In [ ]:
final.shape

In [ ]:
final.head(3)

In [ ]:
final.shape

In [ ]:
final.to_csv(r'C:\Users\2097770\OneDrive - Cognizant\Desktop\Data Analytics Course\Data Analystics Projects\Uber\PreProcessedData/uber_full_data.csv',index = False)

# Hourly Rush via Animated Spatial Analysis

In [ ]:
[final]

In [ ]:
final.dtypes

In [ ]:
final['Date/Time'] = pd.to_datetime(final['Date/Time'])

In [ ]:
final.dtypes

In [ ]:
final['Hour'] = final['Date/Time'].dt.hour 

In [ ]:
final

In [ ]:
final.columns

In [ ]:
hourly_data = []
for h in range(24):
    temp = final[final['Hour'] == h][['Lat','Lon']]
    hourly_data.append(temp.values.tolist())

In [ ]:
len(hourly_data)

In [ ]:
!pip install folium

In [ ]:
import folium 
from folium.plugins import heat_map_withtime as h

In [ ]:
m = folium.Map(
    location = [40.7128, -74.0060],
    zoom_start = 11,
    titles = 'cartodbpositron'
)

In [ ]:
m

In [ ]:
h.HeatMapWithTime(
    hourly_data,
    radius=9,
    auto_play=False,
    max_opacity=0.8
).add_to(m)

In [ ]:
m